# Taxonomy / Topic Clustering Pipeline — Component Demo

Example notebook แสดง component หลักของ taxonomy/clustering pipeline (อ้างอิงจาก `old-concept-code/`):

1. **Input** — `list[str]`
2. **Chunking** — split ข้อความก่อน embed ด้วย `llama-index` `SentenceSplitter`
3. **Embedding** — `BAAI/bge-m3` (sentence-transformers) พร้อม prompt สำหรับจัดหมวดหมู่
4. **Dimensionality reduction** — เลือก UMAP parameters ที่เหมาะสมด้วย grid search แล้ว reduce เหลือ 2 มิติ (`coordinate_xy`)
5. **Clustering** — `KDEWatershedClusterer` (KDE + watershed segmentation) บน `coordinate_xy`
6. **Cluster naming** — ตั้งชื่อกลุ่มด้วย LLM ผ่าน `dspy`

Flow: `texts` → `chunks` (split) → `vectors` (embed) → `coordinate_xy` (UMAP) → `labels` (cluster) → `cluster_name`. Embed เกิดขึ้น **ครั้งเดียว** บน `chunks` — ไม่ embed `texts` ต้นฉบับซ้ำ

แต่ละ section รันแยกอิสระได้ ปรับ parameter/prompt ตามชุดข้อมูลจริงของคุณ

In [ ]:
%pip install -q sentence-transformers umap-learn scikit-learn scikit-image scipy pandas numpy matplotlib tqdm dspy-ai llama-index-core

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(42)

## 1. Input — `list[str]`

Pipeline รับ input เป็น `list[str]` เท่านั้น ตัวอย่างด้านล่างใช้ข้อความเหตุการณ์/ปัญหาภาษาไทยสั้น ๆ แทน sample data — แทนที่ด้วยข้อมูลจริงของคุณ (เช่นโหลดจาก CSV/Parquet แล้ว `.tolist()`)

In [ ]:
# ตัวอย่าง: โหลดจากไฟล์จริง
# df = pd.read_csv("./data/incidents.csv")
# texts: list[str] = df["content"].dropna().astype(str).tolist()

texts: list[str] = [
    "น้ำท่วมฉับพลันในพื้นที่ลุ่มต่ำหลังฝนตกหนักต่อเนื่อง 3 วัน",
    "ระดับน้ำในแม่น้ำเพิ่มสูงขึ้นจนล้นตลิ่งเข้าท่วมบ้านเรือนประชาชน",
    "ไฟไหม้โรงงานอุตสาหกรรมย่านชานเมือง ควันลอยปกคลุมพื้นที่กว้าง",
    "เพลิงไหม้อาคารพาณิชย์กลางเมือง เจ้าหน้าที่เร่งควบคุมสถานการณ์",
    "แผ่นดินไหวขนาด 5.2 สร้างความเสียหายแก่อาคารเก่าหลายหลัง",
    "เกิดอาฟเตอร์ช็อกต่อเนื่องหลังแผ่นดินไหวใหญ่เมื่อคืนที่ผ่านมา",
    "ดินถล่มปิดเส้นทางคมนาคมสายหลักบนภูเขาหลังฝนตกหนัก",
    "พื้นที่เสี่ยงดินสไลด์ถูกประกาศเป็นเขตอันตรายชั่วคราว",
    "ภัยแล้งรุนแรงทำให้ปริมาณน้ำในเขื่อนลดต่ำสุดในรอบ 10 ปี",
    "เกษตรกรได้รับผลกระทบหนักจากภาวะขาดแคลนน้ำเพื่อการเพาะปลูก",
]

print(f"n = {len(texts)}")
texts[:3]

## 2. Chunking — `llama-index` (ก่อนเข้า embedding)

Split `texts` เป็น `chunks` ด้วย `SentenceSplitter` ของ `llama-index` (`llama-index-core`) ก่อน embed เพื่อไม่ให้เกิน context/embedding limit และให้แต่ละจุดใน embedding space สื่อความหมายเดียวชัดเจน

Sample texts ด้านบนสั้นอยู่แล้วจึงได้ 1 chunk/doc — สำหรับข้อความยาวจริงปรับ `CHUNK_SIZE`/`CHUNK_OVERLAP` ตามความเหมาะสม (หน่วยเป็น token) `chunk_doc_ids` เก็บ mapping กลับไปยัง index ของ `texts` ต้นฉบับ (เผื่อหลาย chunk มาจาก doc เดียวกัน)

**ขั้นตอนถัดไปทั้งหมด (embedding → UMAP → clustering → naming) ใช้ `chunks` เท่านั้น — ไม่ยุ่งกับ `texts` อีก**

In [ ]:
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

CHUNK_SIZE = 200
CHUNK_OVERLAP = 20

splitter = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
documents = [Document(text=t, id_=str(i)) for i, t in enumerate(texts)]
nodes = splitter.get_nodes_from_documents(documents)

chunks: list[str] = [n.get_content() for n in nodes]
chunk_doc_ids: list[int] = [int(n.ref_doc_id) for n in nodes]

print(f"{len(texts)} docs -> {len(chunks)} chunks")
chunks[:3]

## 3. Embedding — `BAAI/bge-m3`

ใช้ `sentence-transformers` โหลด `BAAI/bge-m3` แล้ว encode `chunks` เป็น vector พร้อม instruction prompt ที่ชี้นำโมเดลว่าจะ embed เพื่อจุดประสงค์ใด (จัดหมวดหมู่ตามประเภทของปัญหา/ภัยพิบัติ)

**`EMBED_PROMPT` ปรับได้อิสระตามโดเมนข้อมูลของคุณ** — ตัวอย่างด้านล่างคือ prompt เริ่มต้น

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "BAAI/bge-m3"

# --- ผู้ใช้ปรับ prompt ตรงนี้ได้อิสระ ---
EMBED_PROMPT = "จงพิจารณาข้อความนี้เพื่อจัดหมวดหมู่ตามประเภทของปัญหาหรือภัยพิบัติ: "

# ปรับ EMBED_DEVICE ได้ตรง ๆ ("cuda", "cuda:0", "cpu") ถ้าไม่ตั้งจะ auto-detect + fallback เป็น cpu
# ให้เองถ้าโหลดขึ้น GPU แล้วเจอ error ระดับ kernel (เช่น GPU รุ่นเก่าที่ torch build ปัจจุบันไม่รองรับ
# เช่น "CUDA error: no kernel image is available for execution on the device")
EMBED_DEVICE = os.environ.get("EMBED_DEVICE")

if EMBED_DEVICE is None:
    EMBED_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

try:
    embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=EMBED_DEVICE)
    if EMBED_DEVICE.startswith("cuda"):
        # sanity check: ยิง forward เล็ก ๆ ให้ error (ถ้ามี) โผล่ตรงนี้ ไม่ใช่กลางลูป embed_texts
        embed_model.encode(["ทดสอบ"], show_progress_bar=False)
except RuntimeError as e:
    print(f"⚠️ โหลดโมเดลบน {EMBED_DEVICE} ไม่สำเร็จ ({e}) — fallback ไปใช้ cpu แทน")
    EMBED_DEVICE = "cpu"
    embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=EMBED_DEVICE)

print("embed device:", EMBED_DEVICE)

In [ ]:
def embed_texts(
    texts: list[str],
    model: SentenceTransformer,
    prompt: str,
    batch_size: int = 32,
) -> np.ndarray:
    all_vectors = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i : i + batch_size]
        vectors = model.encode(
            batch,
            prompt=prompt,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        all_vectors.extend(vectors)
    return np.array(all_vectors)


vectors = embed_texts(chunks, embed_model, EMBED_PROMPT)
print("Embedding shape:", vectors.shape)

## 4. UMAP parameter selection → `coordinate_xy`

Grid search หา `n_neighbors` / `min_dist` ที่เหมาะสม อิงตาม `UmapClustering._parallel_worker_func` ใน `old-concept-code/main.py`: รันแต่ละชุด parameter ผ่าน UMAP แล้วประเมินด้วย quick clustering (KDE + watershed, section 5) รวม 3 metric เข้าด้วยกัน:

- `silhouette_score` (ยิ่งสูงยิ่งดี — จุดในกลุ่มเดียวกันอยู่ใกล้กัน กลุ่มต่างกันอยู่ห่างกัน)
- `davies_bouldin_score` (ยิ่งต่ำยิ่งดี — ลบออกจาก score)
- `entropy` ของ KDE density histogram (ยิ่งสูงยิ่งดี — ความหนาแน่นกระจายเป็นหลายยอดชัดเจน ไม่ใช่ก้อนเดียวหรือแบนราบไร้โครงสร้าง)

`ranking_score = entropy + silhouette - davies_bouldin` — ใช้จัดอันดับหา parameter ที่ดีที่สุด (เหมือน `ranking_score` ใน `_parallel_worker_func`)

**ต้องรัน cell ของ `KDEWatershedClusterer` (section 5) ก่อน** เพราะใช้ประเมินคุณภาพแต่ละชุด UMAP parameter

In [ ]:
import numpy as np
from scipy.stats import gaussian_kde
from scipy.ndimage import maximum_filter
from skimage.segmentation import watershed


class KDEWatershedClusterer:
    """KDE density + watershed segmentation clusterer. อ้างอิงจาก old-concept-code/clustering.py"""

    def __init__(self, bw_method=0.1, grid_size=100, neighborhood_size=5, rel_threshold=None):
        self.bw_method = bw_method
        self.grid_size = grid_size
        self.neighborhood_size = neighborhood_size
        self.rel_threshold = rel_threshold if rel_threshold is not None else self.bw_method
        self.embedding_, self.kde_dict_, self.peaks_, self.cluster_centers_, self.labels_ = [None] * 5

    def fit_predict(self, X):
        self.embedding_ = X
        x, y = X[:, 0], X[:, 1]
        self.kde_dict_ = self._calculate_kde(x, y)
        self.peaks_ = self._find_peaks()
        if self.peaks_:
            self.cluster_centers_ = np.array([[p["x"], p["y"]] for p in self.peaks_])
        else:
            self.cluster_centers_ = np.empty((0, 2))
        self.labels_ = self._assign_labels_by_watershed(X)
        return self.labels_

    def get_results_dict(self):
        if self.labels_ is None:
            raise RuntimeError("ต้องรัน .fit_predict(X) ก่อน")
        centers_dict = {str(i): c.tolist() for i, c in enumerate(self.cluster_centers_)}
        serializable_kde = {k: v.tolist() if isinstance(v, np.ndarray) else v for k, v in self.kde_dict_.items()}
        return {"labels": self.labels_.tolist(), "centers": centers_dict, "kde_dict": serializable_kde, "xy": self.embedding_.tolist()}

    def _calculate_kde(self, x, y):
        kde = gaussian_kde(np.vstack([x, y]), bw_method=self.bw_method)
        xmin, xmax, ymin, ymax = x.min() - 1, x.max() + 1, y.min() - 1, y.max() + 1
        x_grid, y_grid = np.meshgrid(np.linspace(xmin, xmax, self.grid_size), np.linspace(ymin, ymax, self.grid_size))
        kde_values = kde(np.vstack([x_grid.ravel(), y_grid.ravel()])).reshape(x_grid.shape)
        return {"x_grid": x_grid, "y_grid": y_grid, "kde_values": kde_values, "bw_method": self.bw_method}

    def _find_peaks(self):
        vals = self.kde_dict_["kde_values"]
        mask = (vals == maximum_filter(vals, size=self.neighborhood_size)) & (vals > vals.max() * self.rel_threshold)
        rows, cols = np.where(mask)
        if len(rows) == 0:
            return []
        return [{"x": self.kde_dict_["x_grid"][r, c], "y": self.kde_dict_["y_grid"][r, c]} for r, c in zip(rows, cols)]

    def _assign_labels_by_watershed(self, xy):
        if not self.peaks_:
            return np.full(xy.shape[0], -1)
        kde_values = self.kde_dict_["kde_values"]
        xmin, xmax = self.kde_dict_["x_grid"][0, 0], self.kde_dict_["x_grid"][0, -1]
        ymin, ymax = self.kde_dict_["y_grid"][0, 0], self.kde_dict_["y_grid"][-1, 0]
        markers_grid = np.zeros_like(kde_values, dtype=int)
        for i, peak in enumerate(self.peaks_):
            r = np.abs(self.kde_dict_["y_grid"][:, 0] - peak["y"]).argmin()
            c = np.abs(self.kde_dict_["x_grid"][0, :] - peak["x"]).argmin()
            markers_grid[r, c] = i + 1
        labels_grid = watershed(-kde_values, markers_grid, mask=np.ones_like(kde_values, dtype=bool))
        cols = np.clip(((xy[:, 0] - xmin) / (xmax - xmin) * (self.grid_size - 1)).astype(int), 0, self.grid_size - 1)
        rows = np.clip(((xy[:, 1] - ymin) / (ymax - ymin) * (self.grid_size - 1)).astype(int), 0, self.grid_size - 1)
        return labels_grid[rows, cols] - 1

In [ ]:
import umap
from scipy.stats import entropy as scipy_entropy
from sklearn.metrics import silhouette_score, davies_bouldin_score

# --- grid search space: ปรับตามขนาด/ความหนาแน่นของข้อมูลจริง ---
UMAP_SEARCH_SPACE = {
    "n_neighbors": [5, 10, 15],
    "min_dist": [0.1, 0.3],
}
PROBE_BW = 0.15          # bandwidth ใช้แค่สำหรับ "ประเมิน" คุณภาพแต่ละชุด UMAP param
PROBE_GRID_SIZE = 100
PROBE_NEIGHBORHOOD = 5


def _kde_entropy(kde_values: np.ndarray, bins: int = 1000) -> float:
    """เหมือน _get_entropy_for_worker ใน old-concept-code/main.py"""
    hist, _ = np.histogram(kde_values.flatten(), bins=bins, density=True)
    return scipy_entropy(hist)


param_list = [
    {"n_neighbors": n, "min_dist": d}
    for n in UMAP_SEARCH_SPACE["n_neighbors"]
    for d in UMAP_SEARCH_SPACE["min_dist"]
]

# เก็บ xy / kde_dict / centers ของทุก run ไว้ด้วย ใช้ทั้งจัดอันดับและ plot เปรียบเทียบ top-N ด้านล่าง
# (สำหรับ dataset ใหญ่ ใน main.py ใช้ joblib.Parallel รัน param_list พร้อมกันหลาย core แทน loop ตรง ๆ)
ranking = []
for params in tqdm(param_list, desc="UMAP grid search"):
    reducer = umap.UMAP(n_components=2, metric="cosine", random_state=42, **params)
    xy_probe = reducer.fit_transform(vectors)

    probe_clusterer = KDEWatershedClusterer(
        bw_method=PROBE_BW, grid_size=PROBE_GRID_SIZE, neighborhood_size=PROBE_NEIGHBORHOOD
    )
    probe_labels = probe_clusterer.fit_predict(xy_probe)

    if len(np.unique(probe_labels)) > 1:
        sil = silhouette_score(xy_probe, probe_labels)
        db = davies_bouldin_score(xy_probe, probe_labels)
    else:
        sil, db = -1.0, float("inf")

    entp = _kde_entropy(probe_clusterer.kde_dict_["kde_values"])
    score = entp + sil - db

    ranking.append({
        **params,
        "silhouette": sil,
        "davies_bouldin": db,
        "entropy": entp,
        "score": score,
        "xy": xy_probe,
        "kde_dict": probe_clusterer.kde_dict_,
        "centers": probe_clusterer.cluster_centers_,
    })

ranking.sort(key=lambda r: r["score"], reverse=True)

ranking_df = pd.DataFrame([
    {k: r[k] for k in ("n_neighbors", "min_dist", "silhouette", "davies_bouldin", "entropy", "score")}
    for r in ranking
])
ranking_df

### เปรียบเทียบ candidate อันดับต้น ๆ (visual)

Score สูงสุดไม่ได้แปลว่าดีที่สุดเสมอไปในทางปฏิบัติ — plot contour (density) + scatter ของ top-N candidate เทียบกัน (เหมือน `show_top_n_ranking_plots` ใน `main.py`) เพื่อ eyeball ประกอบการตัดสินใจก่อนเลือก `best_params`

In [ ]:
TOP_N = min(4, len(ranking))
n_cols = 2
n_rows = (TOP_N + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, n_rows * 5), squeeze=False)
axes_flat = axes.flatten()

for i in range(TOP_N):
    r = ranking[i]
    ax = axes_flat[i]
    ax.contourf(r["kde_dict"]["x_grid"], r["kde_dict"]["y_grid"], r["kde_dict"]["kde_values"], levels=20, cmap="Blues", alpha=0.7)
    ax.scatter(r["xy"][:, 0], r["xy"][:, 1], s=10, alpha=0.6)
    if r["centers"].size > 0:
        ax.scatter(r["centers"][:, 0], r["centers"][:, 1], marker="x", s=60, c="red", zorder=10)
    ax.set_title(
        f"#{i+1} n_neighbors={r['n_neighbors']}, min_dist={r['min_dist']}\n"
        f"score={r['score']:.3f} (sil={r['silhouette']:.3f}, db={r['davies_bouldin']:.3f}, ent={r['entropy']:.3f})",
        fontsize=10,
    )
for i in range(TOP_N, len(axes_flat)):
    axes_flat[i].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
best = ranking[0]
best_params = {"n_neighbors": int(best["n_neighbors"]), "min_dist": float(best["min_dist"])}
print("Best UMAP params:", best_params, "| score:", round(best["score"], 4))

# reuse embedding ที่ได้ตอน grid search ตรง ๆ (เหมือน best_embedding ใน UmapClustering.fit) — ไม่ fit ซ้ำ
coordinate_xy = best["xy"]

# fit เก็บ UMAP model object ไว้ด้วย เผื่อต้อง .transform() ข้อมูลใหม่ภายหลัง (เหมือน best_umap_model_ ใน main.py)
best_umap_model = umap.UMAP(n_components=2, metric="cosine", random_state=42, **best_params).fit(vectors)

print("coordinate_xy shape:", coordinate_xy.shape)

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(coordinate_xy[:, 0], coordinate_xy[:, 1], s=20, alpha=0.8)
plt.title(f"UMAP 2D embedding (n_neighbors={best_params['n_neighbors']}, min_dist={best_params['min_dist']})")
plt.xlabel("x")
plt.ylabel("y")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.show()

## 5. Clustering — `KDEWatershedClusterer`

รัน clusterer จริง (คนละ instance จากตอน grid search) บน `coordinate_xy` ที่เลือกแล้ว — ปรับ `bw_method` (bandwidth) ให้เหมาะสมกับความหนาแน่น/จำนวนกลุ่มที่ต้องการ: bandwidth เล็ก → กลุ่มย่อยเยอะ, bandwidth ใหญ่ → กลุ่มใหญ่รวมกัน

In [ ]:
CLUSTER_BW = 0.15
CLUSTER_GRID_SIZE = 120
CLUSTER_NEIGHBORHOOD = 7

clusterer = KDEWatershedClusterer(
    bw_method=CLUSTER_BW, grid_size=CLUSTER_GRID_SIZE, neighborhood_size=CLUSTER_NEIGHBORHOOD
)
labels = clusterer.fit_predict(coordinate_xy)

result_df = pd.DataFrame({
    "text": chunks,
    "doc_id": chunk_doc_ids,
    "x": coordinate_xy[:, 0],
    "y": coordinate_xy[:, 1],
    "cluster_id": labels,
})
print("n clusters (excl. noise=-1):", len([c for c in result_df["cluster_id"].unique() if c >= 0]))
result_df.head()

In [ ]:
kde_dict = clusterer.kde_dict_
centers = clusterer.cluster_centers_

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(kde_dict["x_grid"], kde_dict["y_grid"], kde_dict["kde_values"], levels=20, cmap="Blues", alpha=0.6)
for cid in sorted(c for c in result_df["cluster_id"].unique() if c >= 0):
    pts = result_df[result_df["cluster_id"] == cid]
    ax.scatter(pts["x"], pts["y"], s=30, label=f"cluster {cid}", edgecolor="black", linewidth=0.3)
if centers.size > 0:
    ax.scatter(centers[:, 0], centers[:, 1], marker="x", s=80, c="red", zorder=10, label="center")
ax.set_title(f"KDEWatershedClusterer (bw={CLUSTER_BW})")
ax.legend(loc="best", fontsize=8)
ax.grid(True, linestyle="--", linewidth=0.5)
plt.show()

## 6. Cluster naming — LLM ผ่าน `dspy`

สุ่มตัวอย่างข้อความในแต่ละ `cluster_id` แล้วให้ LLM (ผ่าน `dspy.Signature` + `dspy.Predict`) ตั้งชื่อหมวดหมู่

ปรับ `dspy.LM(...)` เป็น provider/model ของคุณเอง (OpenAI, Azure, local LM, ฯลฯ) — ตัวอย่างด้านล่างใช้ env var `OPENAI_API_KEY` / `DSPY_MODEL`

In [ ]:
import dspy

DSPY_MODEL = os.environ.get("DSPY_MODEL", "openai/gpt-4o-mini")

lm = dspy.LM(DSPY_MODEL, api_key=os.environ.get("OPENAI_API_KEY"))
dspy.configure(lm=lm)


class TopicLabeler(dspy.Signature):
    """วิเคราะห์ข้อความตัวอย่างและตั้งชื่อหมวดหมู่ของปัญหา/ภัยพิบัติที่เกิดขึ้น เป็นภาษาไทยแบบสั้น กระชับ สื่อความหมายถึงภาพรวมของข้อความทั้งหมด"""

    samples = dspy.InputField(desc="รายการข้อความตัวอย่างที่อยู่ในกลุ่มเดียวกัน")
    taxonomy_name = dspy.OutputField(desc="ชื่อหมวดหมู่สั้น กระชับ เป็นภาษาไทย")

In [ ]:
def name_clusters(df: pd.DataFrame, text_col="text", cluster_col="cluster_id", sample_size=15) -> dict[int, str]:
    predictor = dspy.Predict(TopicLabeler)
    cluster_ids = sorted(c for c in df[cluster_col].unique() if c >= 0)
    names: dict[int, str] = {}

    for cid in cluster_ids:
        cluster_texts = df.loc[df[cluster_col] == cid, text_col]
        n = min(len(cluster_texts), sample_size)
        sample = cluster_texts.sample(n=n, random_state=42).tolist()
        samples_str = "\n".join(f"- {t}" for t in sample)

        try:
            pred = predictor(samples=samples_str)
            names[cid] = pred.taxonomy_name.strip(' "\'')
        except Exception as e:
            print(f"cluster {cid} failed: {e}")
            names[cid] = f"cluster_{cid}"

    return names


cluster_names = name_clusters(result_df)
result_df["cluster_name"] = result_df["cluster_id"].map(cluster_names).fillna("noise")
cluster_names

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ax.contourf(kde_dict["x_grid"], kde_dict["y_grid"], kde_dict["kde_values"], levels=20, cmap="Blues", alpha=0.6)
for cid in sorted(c for c in result_df["cluster_id"].unique() if c >= 0):
    pts = result_df[result_df["cluster_id"] == cid]
    ax.scatter(pts["x"], pts["y"], s=30, label=cluster_names.get(cid, str(cid)), edgecolor="black", linewidth=0.3)
if centers.size > 0:
    for i, c in enumerate(centers):
        ax.text(c[0], c[1], cluster_names.get(i, str(i)), fontsize=9, ha="center", zorder=11)
ax.set_title("Named clusters")
ax.legend(loc="best", fontsize=8, bbox_to_anchor=(1.02, 1))
ax.grid(True, linestyle="--", linewidth=0.5)
plt.tight_layout()
plt.show()

result_df

## Summary

`result_df` มีคอลัมน์ `text` (chunk), `doc_id` (index ของ `texts` ต้นฉบับ), `x`, `y`, `cluster_id`, `cluster_name` — output พร้อมใช้เป็น taxonomy/topic map

Component ที่ demo ในนี้ ตรงกับไฟล์ใน `old-concept-code/`:
- Embedding + UMAP search: `taxonomy_01.ipynb`
- `KDEWatershedClusterer`: `clustering.py`
- UMAP grid search / hierarchical (L1/L2) clustering pipeline แบบเต็ม: `main.py` (`UmapClustering`)
- Cluster naming ด้วย LLM: `taxonomy_01.ipynb` (dspy) และ `02-clustering-edit.ipynb` (ollama/gemini)

สำหรับ production หรือ dataset ใหญ่ ให้ดู `main.py::UmapClustering` ซึ่งมี hierarchical clustering (level1/level2), parallel UMAP grid search และ visualization เพิ่มเติม